# **Automated Resume Screening & Skill-Gap Analyzer**

In [28]:
import kagglehub
import pandas as pd
import os
import re
import spacy
import glob

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import joblib
from google.colab import files

# spaCy English model for text processing
nlp = spacy.load("en_core_web_sm")

## **Download dataset**

In [23]:
print("Downloading dataset...")
path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

# Scan recursively for data files
csv_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True) + \
            glob.glob(os.path.join(path, "**/*.CSV"), recursive=True)

if not csv_files:
    raise FileNotFoundError("Could not find any CSV files.")

file_path = csv_files[0]
df = pd.read_csv(file_path)

category_col = [c for c in df.columns if 'category' in c.lower()][0]
resume_col = [c for c in df.columns if 'resume' in c.lower()][0]

print(f"Using '{category_col}' for Category and '{resume_col}' for Resume text.")

Using Colab cache for faster access to the 'resume-dataset' dataset.
Using 'Category' for Category and 'Resume_str' for Resume text.


## **clean dataset**

In [24]:
# fill missing values with empty strings to preserve rows
df[category_col] = df[category_col].fillna("Unknown")
df[resume_col] = df[resume_col].fillna("")

def clean_resume(text):
    text = str(text).lower()
    text = re.sub(r'http\S+\s*', ' ', text)  # Remove links
    text = re.sub(r'\S+@\S+', ' ', text)     # Remove emails
    text = re.sub(r'[^a-z\s]', ' ', text)    # Remove characters/numbers
    text = re.sub(r'\s+', ' ', text).strip() # Clean whitespaces
    return text

print("Cleaning resume text")
df['Cleaned_Resume'] = df[resume_col].apply(clean_resume)

# Standardize column naming
df = df.rename(columns={category_col: 'Category'})

print(f"Total rows loaded: {len(df)}")

Cleaning resume text
Total rows loaded: 2484


## **Example job description and candidate**

In [25]:
# Job Description
job_description = """
We are looking for a Machine Learning Engineer.
Required skills: Python, Machine Learning, TensorFlow, Scikit-learn, Computer Vision, OpenCV, YOLOv8.
Experience with Natural Language Processing (NLP) and Deep Learning is a plus.
"""
cleaned_jd = clean_resume(job_description)

# perfect candidate
perfect_candidate = {
    'Category': 'Data Science (Benchmark)',
    'Resume_str': 'Experienced Software Developer and Machine Learning Engineer. Proficient in Python, TensorFlow, Scikit-learn, and Computer Vision using OpenCV and YOLOv8. Developed mobile fitness apps and published research papers in ML.',
    'Cleaned_Resume': clean_resume('Experienced Software Developer and Machine Learning Engineer. Proficient in Python, TensorFlow, Scikit-learn, and Computer Vision using OpenCV and YOLOv8. Developed mobile fitness apps and published research papers in ML.')
}

In [26]:
df = pd.concat([df, pd.DataFrame([perfect_candidate])], ignore_index=True)

# Vectorization Pipeline
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['Cleaned_Resume'].tolist() + [cleaned_jd])

resume_vectors = tfidf_matrix[:-1]
jd_vector = tfidf_matrix[-1]

# Cosine Similarity
similarity_scores = cosine_similarity(resume_vectors, jd_vector).flatten()
df['Match_Score'] = (similarity_scores * 100).round(2)

# Sort candidates
ranked_resumes = df.sort_values(by='Match_Score', ascending=False)[['Category', 'Match_Score', 'Cleaned_Resume']]

print("\n--- NEW TOP 3 CANDIDATES FOR THE ML ROLE ---")
print(ranked_resumes.head(3)[['Category', 'Match_Score']])


--- NEW TOP 3 CANDIDATES FOR THE ML ROLE ---
                      Category  Match_Score
2484  Data Science (Benchmark)        62.36
2291                      ARTS        16.16
2153                   BANKING        13.20


## **Skill Extraction & Gap Analysis**

In [27]:
# Target parameters
required_skills = ['python', 'machine learning', 'tensorflow', 'scikit-learn', 'computer vision', 'opencv', 'yolo', 'nlp', 'deep learning']

def analyze_skill_gap(resume_text, req_skills):
    doc = nlp(resume_text)
    found_skills = [skill for skill in req_skills if skill in resume_text]
    missing_skills = [skill for skill in req_skills if skill not in resume_text]
    return found_skills, missing_skills

# candidate at the top of the vector score tier
top_candidate_text = ranked_resumes.iloc[0]['Cleaned_Resume']
found, missing = analyze_skill_gap(top_candidate_text, required_skills)

print("\nSKILL AUDIT FOR TOP CANDIDATE")
print(f"Role Category: {ranked_resumes.iloc[0]['Category']}")
print(f"Match Score: {ranked_resumes.iloc[0]['Match_Score']}%")
print(f"Skills Found: {', '.join(found) if found else 'None'}")
print(f"Skills Missing: {', '.join(missing) if missing else 'None'}")


SKILL AUDIT FOR TOP CANDIDATE
Role Category: Data Science (Benchmark)
Match Score: 62.36%
Skills Found: python, machine learning, tensorflow, computer vision, opencv, yolo
Skills Missing: scikit-learn, nlp, deep learning


## **Save**

In [29]:
# Save
model_filename = 'resume_tfidf_vectorizer.joblib'
joblib.dump(vectorizer, model_filename)
print(f"Model assets successfully serialized as {model_filename}!")

# download
files.download(model_filename)

Model assets successfully serialized as resume_tfidf_vectorizer.joblib!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>